In [6]:
import pandas as pd
df_movie = pd.read_excel('movies.xlsx')
df_ott = pd.read_csv('ott.csv')
print(df_movie.head())
print(df_movie.shape)
print(df_ott.head())
print(df_ott.shape)

                            Title  Year  Age  IMDb  Rotten Tomatoes  \
0                       Inception  2010  13+   8.8             0.87   
1                      The Matrix  1999  18+   8.7             0.87   
2          Avengers: Infinity War  2018  13+   8.5             0.84   
3              Back to the Future  1985   7+   8.5             0.96   
4  The Good, the Bad and the Ugly  1966  18+   8.8             0.97   

                        Directors                            Genres  \
0               Christopher Nolan  Action,Adventure,Sci-Fi,Thriller   
1  Lana Wachowski,Lilly Wachowski                     Action,Sci-Fi   
2         Anthony Russo,Joe Russo           Action,Adventure,Sci-Fi   
3                 Robert Zemeckis           Adventure,Comedy,Sci-Fi   
4                    Sergio Leone                           Western   

                        Country                 Language  Runtime  
0  United States,United Kingdom  English,Japanese,French    148.0  
1         

In [9]:
print(df_movie.dtypes)
print(df_ott.dtypes)

Title               object
Year                 int64
Age                    str
IMDb               float64
Rotten Tomatoes    float64
Directors              str
Genres                 str
Country                str
Language               str
Runtime            float64
dtype: object
ID             int64
Netflix        int64
Stan           int64
Prime Video    int64
Disney+        int64
dtype: object


In [11]:
print(df_ott['ID'].is_unique)                                   # should be True
print(df_ott['ID'].min(), df_ott['ID'].max())                   # should be 1 16744
print((df_ott['ID'] == range(1, len(df_ott) + 1)).all())        # True = in order, no gaps

True
1 16744
True


In [12]:
df_movie['ID'] = range(1, len(df_movie) + 1)

df_merged = pd.merge(df_movie, df_ott, on='ID', how='left', indicator=True)

print(df_merged['_merge'].value_counts())   # all 16744 should be 'both'
print(df_merged.shape)                      # expect (16744, 16): 10 + ID + 4 platforms + _merge
print(df_merged.head())

df_merged = df_merged.drop(columns='_merge')

_merge
both          16744
left_only         0
right_only        0
Name: count, dtype: int64
(16744, 16)
                            Title  Year  Age  IMDb  Rotten Tomatoes  \
0                       Inception  2010  13+   8.8             0.87   
1                      The Matrix  1999  18+   8.7             0.87   
2          Avengers: Infinity War  2018  13+   8.5             0.84   
3              Back to the Future  1985   7+   8.5             0.96   
4  The Good, the Bad and the Ugly  1966  18+   8.8             0.97   

                        Directors                            Genres  \
0               Christopher Nolan  Action,Adventure,Sci-Fi,Thriller   
1  Lana Wachowski,Lilly Wachowski                     Action,Sci-Fi   
2         Anthony Russo,Joe Russo           Action,Adventure,Sci-Fi   
3                 Robert Zemeckis           Adventure,Comedy,Sci-Fi   
4                    Sergio Leone                           Western   

                        Country          

In [13]:
pd.concat([df_movie, df_ott], axis=0)

,Title,Year,Age,IMDb,Rotten Tomatoes,Directors,Genres,Country,Language,Runtime,ID,Netflix,Stan,Prime Video,Disney+
0,Inception,2010.0,13+,8.8,0.87,Christopher Nolan,"Action,Adventure,Sci-Fi,Thriller","United States,United Kingdom","English,Japanese,French",148.0,1,NaN,NaN,NaN,NaN
1,The Matrix,1999.0,18+,8.7,0.87,"Lana Wachowski,Lilly Wachowski","Action,Sci-Fi",United States,English,136.0,2,NaN,NaN,NaN,NaN
2,Avengers: Infinity War,2018.0,13+,8.5,0.84,"Anthony Russo,Joe Russo","Action,Adventure,Sci-Fi",United States,English,149.0,3,NaN,NaN,NaN,NaN
3,Back to the Future,1985.0,7+,8.5,0.96,Robert Zemeckis,"Adventure,Comedy,Sci-Fi",United States,English,116.0,4,NaN,NaN,NaN,NaN
4,"The Good, the Bad and the Ugly",1966.0,18+,8.8,0.97,Sergio Leone,Western,"Italy,Spain,West Germany",Italian,161.0,5,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16739,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16740,0.0,0.0,1.0,0.0
16740,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16741,0.0,0.0,1.0,0.0
16741,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16742,0.0,0.0,1.0,0.0
16742,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16743,0.0,0.0,1.0,0.0


In [27]:
# movies per year available on netflix
mo_gpby = (
    df_merged.groupby(['Netflix', 'Year'])
        .agg({'Title':'count'})
    .reset_index()
    .rename(columns={'Title': 'Films'})
)

netflix_2012 = mo_gpby[(mo_gpby['Year'] >= 2012) & (mo_gpby['Netflix'] == 1)]

netflix_2012

,Netflix,Year,Films
199,1,2012,195
200,1,2013,200
201,1,2014,202
202,1,2015,252
203,1,2016,236
204,1,2017,294
205,1,2018,307
206,1,2019,138
207,1,2020,31


"New movies in Netflix has peaked 2017, appears to be tapering off"

In [16]:
mo_runtime = (
    df_merged
    .groupby('Year')
    .agg({'Runtime':'mean'})
    .reset_index()
)
mo_runtime = mo_runtime[mo_runtime['Year'] >=2012]
mo_runtime

,Year,Runtime
100,2012,90.511714
101,2013,91.545940
102,2014,92.995763
103,2015,92.914646
104,2016,93.992140
105,2017,94.460961
106,2018,94.635678
107,2019,93.410413
108,2020,93.976562


"industry seems pretty static with run time, very slight increases may be a result of invcreased advertising/marketing but only an assumption"

In [22]:
mo_reviews = (
    df_merged
    .groupby('Year')
    .agg({'Rotten Tomatoes': ['min', 'max']})
    .reset_index()
)

mo_reviews = mo_reviews[mo_reviews['Year'] >= 2012]

mo_reviews

Year Rotten Tomatoes     
                      min  max
100  2012            0.04  1.0
101  2013            0.02  1.0
102  2014            0.05  1.0
103  2015            0.05  1.0
104  2016            0.02  1.0
105  2017            0.04  1.0
106  2018            0.06  1.0
107  2019            0.05  1.0
108  2020            0.06  1.0

"Films appear to be establishing a better base line in recent years, its a small sample but this may indicate a drop in quality in the coming year"